# Fase 2 · Transformación de datos en los datasets

## Objetivo

El objetivo de esta fase consiste en ejecutar los cambios y ajustes detectados durante el Análisis Exploratorio de Datos (EDA), para unificar, limpiar y transformar los datasets seleccionados en este proyecto.

En esta etapa se trabaja principalmente en:

- abordar los duplicados,
- combinar varios datasets,
- gestionar los valores nulos,
- homogenizar categorías para asegurar la integridad semántica,
- y exportar los archivos finales ya depurados.

Este proceso permitirá tener un conjunto de datos más consistentes, comparables y listos para la siguiente fase de análisis avanzado y visualización.

In [1]:
# Importación de librerías
import pandas as pd
import numpy as np
import re

# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación de módulos de transformación 
from src.etl.load_data import load_friends_data_raw
from src.etl import transform as trans
from transformers import pipeline
from src.etl import column_standardizer as stan

 
# Configuración para visualizar todas las columnas del DataFrame
pd.set_option('display.max_columns', None) 

c:\Users\dacil\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dfs = load_friends_data_raw()

[2026-05-25 10:18:45] INFO - Cargando datasets desde: C:\Users\dacil\Desktop\Adalab\pair\modulo_4_pair\friends-analytics-workflow\data_raw
[2026-05-25 10:18:45] INFO - → Cargando weddings_divorces_ross.csv...
[2026-05-25 10:18:45] INFO - → Cargando friends_cameos.csv...
[2026-05-25 10:18:45] INFO - → Cargando friends_emotions.csv...
[2026-05-25 10:18:45] INFO - → Cargando friends_episodes.csv...
[2026-05-25 10:18:45] INFO - → Cargando friends_sets.csv...
[2026-05-25 10:18:45] INFO - → Cargando friends_info.csv...
[2026-05-25 10:18:45] INFO - → Cargando friends_quotes.csv...
[2026-05-25 10:18:45] INFO - → Cargando friends.csv...
[2026-05-25 10:18:47] INFO - → Cargando phoebe_buffay_songs.csv...
[2026-05-25 10:18:47] INFO - → Cargando duck_and_chicken.csv...
[2026-05-25 10:18:47] INFO - Todos los datasets fueron cargados correctamente.


## 1. Transformación de  las variables numéricas (friends_quotes) de orden de float a entero (int) para mejorar la estructura. 

In [3]:
df_quotes = dfs["quotes"]

df_quotes.head()

,author,episode_number,episode_title,quote,quote_order,season
0,Monica,1.0,Monica Gets A Roommate,There's nothing to tell! He's just some guy I ...,0.0,1.0
1,Joey,1.0,Monica Gets A Roommate,"C'mon, you're going out with the guy! There's ...",1.0,1.0
2,Chandler,1.0,Monica Gets A Roommate,"All right Joey, be nice. So does he have a hum...",2.0,1.0
3,Phoebe,1.0,Monica Gets A Roommate,"Wait, does he eat chalk?",3.0,1.0
4,Phoebe,1.0,Monica Gets A Roommate,"Just, 'cause, I don't want her to go through w...",4.0,1.0


In [4]:
df_quotes["quote_order"] = df_quotes["quote_order"].astype(int)
df_quotes["season"] = df_quotes["season"].astype(int)
df_quotes["episode_number"] = df_quotes["episode_number"].astype(int)


In [5]:
df_quotes.head()

,author,episode_number,episode_title,quote,quote_order,season
0,Monica,1,Monica Gets A Roommate,There's nothing to tell! He's just some guy I ...,0,1
1,Joey,1,Monica Gets A Roommate,"C'mon, you're going out with the guy! There's ...",1,1
2,Chandler,1,Monica Gets A Roommate,"All right Joey, be nice. So does he have a hum...",2,1
3,Phoebe,1,Monica Gets A Roommate,"Wait, does he eat chalk?",3,1
4,Phoebe,1,Monica Gets A Roommate,"Just, 'cause, I don't want her to go through w...",4,1


In [6]:
df_quotes.to_csv("../data_processed/friends_quotes.csv", index=False, encoding="utf-8")

## 2. Limpiar y estandarizar la columna written_by (friends_info)

In [7]:
df_info= dfs["info"]

df_info.head()

,season,episode,title,directed_by,written_by,air_date,us_views_millions,imdb_rating
0,1,1,The Pilot,James Burrows,David Crane & Marta Kauffman,1994-09-22,21.5,8.3
1,1,2,The One with the Sonogram at the End,James Burrows,David Crane & Marta Kauffman,1994-09-29,20.2,8.1
2,1,3,The One with the Thumb,James Burrows,Jeffrey Astrof & Mike Sikowitz,1994-10-06,19.5,8.2
3,1,4,The One with George Stephanopoulos,James Burrows,Alexa Junge,1994-10-13,19.7,8.1
4,1,5,The One with the East German Laundry Detergent,Pamela Fryman,Jeff Greenstein & Jeff Strauss,1994-10-20,18.6,8.5


In [8]:
trans.process_friends_writers(df_info)

¡Fichero corregido con éxito! Guardado en: C:\Users\dacil\Desktop\Adalab\pair\modulo_4_pair\friends-analytics-workflow\data_processed\writters.csv (303 filas).


,season,episode,writter,rol
0,1,1,David Crane,Writer
1,1,1,Marta Kauffman,Writer
2,1,2,David Crane,Writer
3,1,2,Marta Kauffman,Writer
4,1,3,Jeffrey Astrof,Writer
...,...,...,...,...
298,10,16,Ted Cohen,Writer
299,10,17,Marta Kauffman,Writer
300,10,17,David Crane,Writer
301,10,18,Marta Kauffman,Writer


In [9]:
df_info.drop("written_by", axis=1, inplace=True)

df_info.head(2)

,season,episode,title,directed_by,air_date,us_views_millions,imdb_rating
0,1,1,The Pilot,James Burrows,1994-09-22,21.5,8.3
1,1,2,The One with the Sonogram at the End,James Burrows,1994-09-29,20.2,8.1


In [10]:
df_quotes.to_csv("../data_processed/friends_info.csv", index=False, encoding="utf-8")

#### Traducir las columnas

In [11]:
df_dac = dfs["dac"]

In [12]:
df_dac = stan.standardize_columns(df_dac)

In [13]:
df_dac.head()

,season,episode_number,animal,accion
0,3,3x21,Pollito,Joey lo compra
1,3,3x22,Pollito,Joey y Chandler cuidan de él.
2,3,3x22,Pato,Chandler lo rescata para que el pollito tenga ...
3,3,3x25,Pollito,Aparecen en el apartamento de los chicos.
4,3,3x25,Pato,Aparecen en el apartamento de los chicos.


In [14]:
df_dac.to_csv("../data_processed/duck_and_chicken.csv", index=False, encoding="utf-8")

In [15]:
df_cameos = dfs["cameos"]
df_cameos.head(1)

,Actor/Actriz,Personaje,Descripción/Temporada
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel (T8)


In [16]:
# 1. Extraemos la descripción y el número de la temporada usando Regex
# El patrón busca "T" seguido de uno o más números d+ dentro de un paréntesis
df_extracted = df_cameos["Descripción/Temporada"].str.extract(r"(?P<descripcion>.*?)\s*\(T(?P<temporada>\d+)\)")

# 2. Asignamos los resultados de vuelta a nuestro DataFrame original
df_cameos["descripcion"] = df_extracted["descripcion"]
df_cameos["temporada"] = df_extracted["temporada"]

# 3. Borramos la columna vieja que ya no necesitamos
df_cameos = df_cameos.drop(columns=["Descripción/Temporada"])

# Ver el resultado
df_cameos.head(1)

,Actor/Actriz,Personaje,descripcion,temporada
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel,8


In [17]:
df_cameos = stan.standardize_columns(df_cameos)

In [18]:
df_cameos.head()

,actor,author,description,season
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel,8
1,Bruce Willis,Paul Stevens,Padre de Elizabeth y novio de Rachel,6
2,Julia Roberts,Susie Moss,Compañera de primaria de Chandler,2
3,Charlie Sheen,Ryan,Marinero novio de Phoebe que tiene varicela,2
4,Danny DeVito,Roy,El stripper sensible en la despedida de Phoebe,10


In [19]:
df_cameos.to_csv("../data_processed/friends_cameos.csv", index=False, encoding="utf-8")

In [20]:
df_sets = dfs["sets"]

In [21]:
df_sets = stan.standardize_columns(df_sets)

In [22]:
df_sets.head(1)

,stage,type,%_scenes,description,est_num_scenes
0,Apartamento de Monica,Principal,38,El escenario con más tiempo de pantalla (cocin...,1900


In [23]:
df_sets.to_csv("../data_processed/friends_sets.csv", index=False, encoding="utf-8")

In [24]:
df_songs = dfs["songs"]

In [25]:
df_songs = stan.standardize_columns(df_songs)

In [26]:
df_songs.head(1)

,season,episode_number,song,description
0,1,1x01,Your Love,Your love is like a giant pigeon...


In [27]:
df_songs.to_csv("../data_processed/friends_songs.csv", index=False, encoding="utf-8")

In [28]:
df_weddings = dfs["weddings"]

In [29]:
df_weddings = stan.standardize_columns(df_weddings)

In [30]:
df_weddings.head(1)

,author,event,season,detail
0,Carol Willick,Boda,0,Ocurre antes del piloto (flashbacks).


In [31]:
df_weddings.to_csv("../data_processed/friends_weddings_divorce_ross.csv", index=False, encoding="utf-8")

In [32]:
df_episodes = dfs["epiv3"]

In [33]:
df_episodes = stan.standardize_columns(df_episodes)

In [34]:
df_episodes.head(1)

,year_of_prod,season,episode_number,episode_title,duration,summary,director,stars,votes
0,1994,1,1,The One Where Monica Gets a Roommate: The Pilot,22,"Monica and the gang introduce Rachel to the ""r...",James Burrows,8.3,7440


In [35]:
df_episodes.to_csv("../data_processed/friends_episodes.csv", index=False, encoding="utf-8")

In [36]:
df_emotions = dfs["emotions"]

In [37]:
df_emotions = stan.standardize_columns(df_emotions)

In [38]:
df_emotions.head(1)

,season,episode_number,scene,utterance,emotion
0,1,1,4,1,Mad


In [39]:
df_emotions.to_csv("../data_processed/friends_emotions.csv", index=False, encoding="utf-8")

In [40]:
df_friends = dfs["friends"]

In [41]:
df_friends = stan.standardize_columns(df_friends)

In [42]:
df_friends.head(1)

,text,speaker,season,episode_number,scene,utterance
0,There's nothing to tell! He's just some guy I ...,Monica Geller,1,1,1,1


In [43]:
df_friends.to_csv("../data_processed/friends.csv", index=False, encoding="utf-8")